In [5]:
import math
def block_stats(n, B):
    bs = [min(B, n-k) for k in range(0, n, B)]  # [128, 128, 128, ...]
    local_pairs = sum(b*(b-1)//2 for b in bs)  # 批内有几次更新 (每列单独计算，各批加在一起)，可以理解为FLOPs/（2d_{out}）
    outer_pairs = sum(b*(n-sum(bs[:t+1])) for t,b in enumerate(bs))  # 批外有几次更新 (每列单独计算，各批加在一起)，可以理解为FLOPs/（2d_{out}）
    weight_outer_visits = sum(n-sum(bs[:t+1]) for t in range(len(bs)))  # 批外权重访问次数（每列单独计算）。
    # 例B=128，d_in=1024，第一个batch结束后，有896列权重要接受来自128列权重的更新。但并不需要执行128次读写，只需要1次读写。因此是896，而不是896*128
    return {
        "blocks":len(bs),
        "inner_calls": sum(b-1 for b in bs),
        "outer_calls":len(bs)-1,
        "local_pairs_per_row":local_pairs,
        "outer_pairs_per_row":outer_pairs,
        "logical_w_visits_per_row":local_pairs+weight_outer_visits  # 权重访问次数（每列单独计算）对于批内，更新次数就是访问次数。
    }
for B in (1,32,128,256,512,1024):
    x=block_stats(1024,B)
    print(B, x, "avg touches:",x['logical_w_visits_per_row']/1024)
assert all(block_stats(1024,B)["local_pairs_per_row"]+block_stats(1024,B)["outer_pairs_per_row"]==1024*1023//2 for B in (1,3,32,128,256,512,1024))
print("percent time:",(2026.801/1905.424-1)*100,(4408.545/4082.185-1)*100)


1 {'blocks': 1024, 'inner_calls': 0, 'outer_calls': 1023, 'local_pairs_per_row': 0, 'outer_pairs_per_row': 523776, 'logical_w_visits_per_row': 523776} avg touches: 511.5
32 {'blocks': 32, 'inner_calls': 992, 'outer_calls': 31, 'local_pairs_per_row': 15872, 'outer_pairs_per_row': 507904, 'logical_w_visits_per_row': 31744} avg touches: 31.0
128 {'blocks': 8, 'inner_calls': 1016, 'outer_calls': 7, 'local_pairs_per_row': 65024, 'outer_pairs_per_row': 458752, 'logical_w_visits_per_row': 68608} avg touches: 67.0
256 {'blocks': 4, 'inner_calls': 1020, 'outer_calls': 3, 'local_pairs_per_row': 130560, 'outer_pairs_per_row': 393216, 'logical_w_visits_per_row': 132096} avg touches: 129.0
512 {'blocks': 2, 'inner_calls': 1022, 'outer_calls': 1, 'local_pairs_per_row': 261632, 'outer_pairs_per_row': 262144, 'logical_w_visits_per_row': 262144} avg touches: 256.0
1024 {'blocks': 1, 'inner_calls': 1023, 'outer_calls': 0, 'local_pairs_per_row': 523776, 'outer_pairs_per_row': 0, 'logical_w_visits_per_row

In [11]:
import math
print(f"opt 125M: 线性层d_in={768}/{3072}, \sqrt(d_in) = {math.sqrt(768)}/{math.sqrt(3072)}")
print(f"opt 13B: 线性层d_in={5120}/{20480}, \sqrt(d_in) = {math.sqrt(5120)}/{math.sqrt(20480)}")
print(f"opt 30B: 线性层d_in={7168}/{28672}, \sqrt(d_in) = {math.sqrt(7168)}/{math.sqrt(28672)}")
print(f"opt 66B: 线性层d_in={9216}/{36864}, \sqrt(d_in) = {math.sqrt(9216)}/{math.sqrt(36864)}")
print(f"opt 175B: 线性层d_in={12288}/{49152}, \sqrt(d_in) = {math.sqrt(12288)}/{math.sqrt(49152)}")

opt 125M: 线性层d_in=768/3072, \sqrt(d_in) = 27.712812921102035/55.42562584220407
opt 13B: 线性层d_in=5120/20480, \sqrt(d_in) = 71.55417527999327/143.10835055998655
opt 30B: 线性层d_in=7168/28672, \sqrt(d_in) = 84.6640419540669/169.3280839081338
opt 66B: 线性层d_in=9216/36864, \sqrt(d_in) = 96.0/192.0
opt 175B: 线性层d_in=12288/49152, \sqrt(d_in) = 110.85125168440814/221.70250336881628


In [6]:
import math

print(math.sqrt(2 * 12288))

156.76734353812338
